## Knowledge Distillation on MNIST

We first train a larger teacher network on MNIST using ordinary cross-entropy.
Then we train the same small student architecture in two ways:

1. **Hard-label baseline**: train on the true digit labels only  
2. **Distilled student**: train on both the true labels and the teacher's softened output distribution

The key idea is that the teacher's full probability distribution contains more information than a one-hot label.
For example, a teacher may predict that an image is mostly a "3", but also somewhat resembles an "8".
The student can learn from that structure and often generalizes better than if it were trained on hard labels alone.

We train a strong teacher first. Then we freeze it. For each training image, the student sees both the correct label and the teacher’s probability distribution over all classes. The hard label tells the student what is correct. The soft target tells the student how the teacher organizes similarity between classes.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Narrative

MNIST images are 28x28 grayscale digits. Since you want a very simple demo, we will flatten each image into a vector of length 784 and use fully connected networks only.

In [ ]:
transform = transforms.ToTensor()

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

We are not doing anything fancy with preprocessing. This helps keep the focus on distillation rather than data engineering.

In [ ]:

class TeacherNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 1200),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1200, 1200),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1200, 10)
        )

    def forward(self, x):
        return self.net(x)


class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.net(x)


The teacher is a larger FCN with more hidden units and an extra hidden layer.
The student is intentionally much smaller.

That gives you a clean story:

teacher = more expressive, more accurate, more cumbersome

student = cheaper, simpler, easier to deploy

In [ ]:

@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    return correct / total

We will compare three models:

teacher

student trained normally on hard labels

student trained with distillation

In [ ]:

def train_supervised(model, loader, epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)

        avg_loss = running_loss / len(loader.dataset)
        acc = evaluate_accuracy(model, test_loader)
        print(f"Epoch {epoch:2d} | loss={avg_loss:.4f} | test_acc={acc:.4f}")

    return model




This is the ordinary baseline training loop using cross-entropy against the true labels only.

You will use this both for the teacher and for the hard-label student baseline.

Distillation loss

In [ ]:

def distillation_loss(student_logits, teacher_logits, y_true, alpha=0.5, temperature=4.0):
    """
    alpha: weight on hard-label loss
    temperature: softmax temperature for distillation
    """
    hard_loss = F.cross_entropy(student_logits, y_true)
    student_log_probs_t = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs_t = F.softmax(teacher_logits / temperature, dim=1)

    soft_loss = F.kl_div(
        student_log_probs_t,
        teacher_probs_t,
        reduction="batchmean"
    ) * (temperature ** 2)

    return alpha * hard_loss + (1 - alpha) * soft_loss


This is the core of the demo.

The student learns from two sources:

the true label via ordinary cross-entropy

the teacher’s softened class distribution via KL divergence

The temperature**2 factor is standard in KD implementations because it keeps the relative scale of the gradients sensible when temperature is used.

Even if you do not want to dwell on temperature in the presentation, this is the cleanest correct form to use.

Distillation training loop

In [ ]:

def train_student_kd(student, teacher, loader, epochs=5, lr=1e-3, alpha=0.5, temperature=4.0):
    student = student.to(device)
    teacher = teacher.to(device)

    teacher.eval()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        student.train()
        running_loss = 0.0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            with torch.no_grad():
                teacher_logits = teacher(xb)

            student_logits = student(xb)

            loss = distillation_loss(
                student_logits=student_logits,
                teacher_logits=teacher_logits,
                y_true=yb,
                alpha=alpha,
                temperature=temperature
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)

        avg_loss = running_loss / len(loader.dataset)
        acc = evaluate_accuracy(student, test_loader)
        print(f"Epoch {epoch:2d} | loss={avg_loss:.4f} | test_acc={acc:.4f}")

    return student




The teacher is frozen here. We are not updating it.
We only use its logits to guide the student.

That is the cleanest way to explain KD:

train teacher first

freeze teacher

train student to match labels and teacher behavior

Train Teacher

In [ ]:

teacher = TeacherNet()
teacher = train_supervised(teacher, train_loader, epochs=5, lr=1e-3)

teacher_acc = evaluate_accuracy(teacher, test_loader)
print(f"\nFinal teacher test accuracy: {teacher_acc:.4f}")




Train the student baseline on hard labels only

In [ ]:
student_hard = StudentNet()
student_hard = train_supervised(
    student_hard, 
    train_loader, 
    epochs=5, 
    lr=1e-3
)

student_hard_acc = evaluate_accuracy(student_hard, test_loader)
print(f"\nFinal hard-label student test accuracy: {student_hard_acc:.4f}")

This is your control group.

It answers the question: how well does the small model do if we train it the usual way?

In [ ]:

student_kd = StudentNet()

student_kd = train_student_kd(
    student=student_kd,
    teacher=teacher,
    loader=train_loader,
    epochs=5,
    lr=1e-3,
    alpha=0.5,
    temperature=4.0
)

student_kd_acc = evaluate_accuracy(student_kd, test_loader)
print(f"\nFinal KD student test accuracy: {student_kd_acc:.4f}")

This is the main comparison.

If distillation is helping, student_kd should usually beat student_hard, even though both students have the same architecture.

That is the point you want your audience to remember.

In [ ]:

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model comparison")
print("-" * 50)
print(f"Teacher params:           {count_params(teacher):,}")
print(f"Student params:           {count_params(student_hard):,}")
print()
print(f"Teacher test accuracy:    {teacher_acc:.4f}")
print(f"Student hard accuracy:    {student_hard_acc:.4f}")
print(f"Student KD accuracy:      {student_kd_acc:.4f}")

12. Optional: inspect soft targets for one example

In [ ]:

@torch.no_grad()
def show_teacher_probs(model, dataset, idx=0, temperature=4.0):
    model.eval()
    x, y = dataset[idx]
    x = x.unsqueeze(0).to(device)

    logits = model(x)
    probs = F.softmax(logits, dim=1).squeeze().cpu()
    probs_t = F.softmax(logits / temperature, dim=1).squeeze().cpu()

    print(f"True label: {y}\n")
    print("Standard probabilities:")
    for i, p in enumerate(probs.tolist()):
        print(f"  class {i}: {p:.4f}")

    print(f"\nSoftened probabilities (T={temperature}):")
    for i, p in enumerate(probs_t.tolist()):
        print(f"  class {i}: {p:.4f}")

In [ ]:

show_teacher_probs(teacher, test_ds, idx=0, temperature=4.0)



This is useful on a slide because it shows the idea visually.

The hard label says only:

this image is a 7

The teacher distribution says something richer:

this is mostly a 7

but if I had to confuse it, 9 is more plausible than 1

and 1 is more plausible than 6

That extra structure is what the student learns.

In the context of large language models, knowledge distillation works by training a smaller or cheaper model to imitate the behavior of a stronger model rather than learning directly from raw human-labeled data. The larger model (the “teacher,” such as GPT-4–class systems) generates responses to a wide range of prompts. These responses implicitly encode a great deal of structure: reasoning style, formatting, preferred explanations, and patterns of language. A smaller model (the “student”) is then trained on those prompt–response pairs so that it learns to produce outputs similar to the teacher’s. In effect, the student is learning the teacher’s mapping from prompts to responses, compressing the capabilities of a very large model into a more efficient one. In modern LLM training pipelines this is often combined with other methods—such as reinforcement learning from human feedback and additional curated datasets—but the distillation component allows much of the teacher model’s behavior to be transferred into a model that is cheaper to run and easier to deploy.